<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 10 · RLHF

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/10-rlhf.ipynb)

**From Traditional NLP to Modern LLMs** · Day 3 · 120 minutes (55 briefing, 55 lab, 10 debrief)

The full reinforcement-learning-from-human-feedback pipeline, DPO, and the ways it goes wrong.

**By the end of this notebook you can:**

- Describe the three-stage RLHF pipeline
- Optimize a small LM against a reward model with a KL constraint
- Apply DPO and compare
- Name RLHF's failure modes and observe one

**Stack:** PyTorch, Hugging Face

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "10-rlhf"
_CONTENT_SHA = "7d5d537fe8ed1d83"
_EXERCISES = {1: ('response_logprobs',), 2: ('token_rewards',), 3: ('exact_kl',), 5: ('dpo_loss',)}
_HINTS = {1, 2, 3, 5}  # exercises with a folded Hint cell above their TODO cell
_PATHS = {'offline': 'offline stand-ins and test doubles: checks that the code runs, not what a model does', 'open': 'real open models from the Hugging Face Hub, no API keys', 'keyed': 'commercial APIs with keys'}
_FALLBACK_NOTE = 'On a CPU runtime, FAST runs short smoke trainings of the same GPT-2; the trained-policy checkpoints are skipped, so reward hacking is not shown'
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_OFFLINE_FLAGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
            if n in _HINTS:
                print(f"[workshop] Stuck? Open the folded Hint above TODO {n} first.")
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Read the assertion above,"
                f" fix TODO {n}, or go on with workshop.use_reference({n!r})."
            )
            if n in _HINTS:
                print(f"[workshop] The folded Hint above TODO {n} says what to check.")

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        path = self.path()
        # _PATHS comes from readiness.paths in _variables.yml; the offline entry says that
        # such numbers check the code, not a model.
        print(f"  What ran: {_PATHS[path]}.")
        if path == "open" and _FALLBACK_NOTE:
            print(f"  Read the numbers with this in mind: {_FALLBACK_NOTE}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def path(self):
        """offline, keyed or open: the same rule as scripts/add_run_record.py's path_of."""
        ns = _get_ipython().user_ns
        if self.fallbacks or any(_os.environ.get(flag) for flag in _OFFLINE_FLAGS):
            return "offline"
        if ns.get("PROVIDER") in ("openai", "anthropic") or ns.get("JEV_PATH") == "keyed":
            return "keyed"
        return "open"

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

**Runtime.** Use a GPU: *Runtime → Change runtime type → T4 GPU*. This lab fine-tunes an 82-million-parameter GPT-2 three times. On a CPU runtime it switches itself to a `FAST` run: every unit checkpoint still runs, each training becomes a short smoke test of a few steps, and the checkpoints about trained policies are skipped with a printed message. No API key is needed.

**What you build.** The policy-optimization stage of RLHF (briefing, sections 3 and 4): the log-probabilities of a response, the per-token reward with a KL penalty, and the exact KL divergence that measures drift. A provided loop then fine-tunes GPT-2 against the reward model you trained in Lab 9, with the penalty and without it. Finally you write the DPO loss (section 6) and train the same model on the same preference pairs without a reward model.

**How the lab works.** There is a Step 0 with nothing to write, five exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one short function;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your code is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

**What you load.** Lab 6's small GPT-2 (`distilbert/distilgpt2`) as the reference policy, and three files from Lab 9: the prompts, the preference pairs and the reward model. Nothing later in the workshop needs anything from this lab.

In [ ]:
# Setup: pinned installs, imports, seeds, device, run size and hyperparameters.
import os

# Deterministic cuBLAS kernels, so that a policy and its exact copy give bit-identical
# log-probabilities on a GPU (Exercise 1). This must be set before CUDA starts.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import importlib.metadata
import sys

# Versions this lab was written against (Colab's transformers on 2026-10-04, as found for Lab 7).
# torch is not pinned: reinstalling it on Colab takes minutes and the lab uses only stable APIs.
PINS = {"transformers": "5.18.0"}


def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


_needed = [f"{name}=={version}" for name, version in PINS.items() if installed_version(name) != version]
if _needed:
    %pip install -q {" ".join(_needed)}

import copy
import gzip
import hashlib
import io
import json
import math
import random
import re
import textwrap
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

transformers.logging.set_verbosity_error()  # hide routine warnings
transformers.utils.logging.disable_progress_bar()  # no download and weight-loading bars

# The seed protocol (scripts/lab10_seed_protocol.py) sets these variables. Participants: leave them.
SEED = int(os.environ.get("NLP_LLMS_LAB10_SEED", "0"))  # training draws, samples, DPO data order
PROTOCOL = os.environ.get("NLP_LLMS_LAB10_PROTOCOL") == "1"  # record failed checks instead of stopping
SKIP_STRETCH = os.environ.get("NLP_LLMS_LAB10_SKIP_STRETCH") == "1"  # the protocol's --skip-stretch
EVAL_SEED = 1234  # one fixed evaluation generator, shared by every policy and every seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.use_deterministic_algorithms(True, warn_only=True)
IN_COLAB = "google.colab" in sys.modules

# TEST-ONLY SWITCH. Leave it unset. The repository's tests set NLP_LLMS_OFFLINE_TINY=1 (or
# NLP_LLMS_LAB10_OFFLINE) on machines that cannot reach huggingface.co. The lab then runs on a
# RANDOMLY INITIALIZED stand-in policy, a stand-in vocabulary, a stand-in reward model and
# synthetic pairs (see the hidden cell under "Data and model"). Every number printed in that mode
# is marked and is not a result.
OFFLINE = os.environ.get("NLP_LLMS_LAB10_OFFLINE", "tiny" if os.environ.get("NLP_LLMS_OFFLINE_TINY") == "1" else "")
assert OFFLINE in ("", "tiny", "distilgpt2-shape"), OFFLINE
NOTE = "   [TEST-ONLY random stand-in: not a result]" if OFFLINE else ""

FAST = DEVICE == "cpu" and OFFLINE != "tiny"  # the CPU smoke run; set to False to force the full run
TRAINED_CHECKS = not FAST and not OFFLINE  # checkpoints about trained policies need the full run
# Test-only: NLP_LLMS_LAB10_FORCE_CHECKS=1 with NLP_LLMS_LAB10_PROTOCOL=1 runs those checks on a stand-in
# too (recording, not raising, their failures), so that the repository's tests execute every line.
TRAINED_CHECKS = TRAINED_CHECKS or os.environ.get("NLP_LLMS_LAB10_FORCE_CHECKS") == "1"

# Run sizes. FULL is the T4 path. FAST is the CPU path. TINY is the test-only stand-in.
if OFFLINE == "tiny":
    RUN = dict(mode="tiny", batch=16, steps=20, log_every=5, eval_prompts=64, eval_samples=8,
               dpo_batch=32, dpo_train_pairs=None, dpo_heldout_pairs=None, sweep_steps=10)
elif FAST:
    RUN = dict(mode="fast", batch=8, steps=6, log_every=2, eval_prompts=8, eval_samples=2,
               dpo_batch=8, dpo_train_pairs=48, dpo_heldout_pairs=64, sweep_steps=0)
else:
    RUN = dict(mode="full", batch=64, steps=150, log_every=10, eval_prompts=64, eval_samples=8,
               dpo_batch=32, dpo_train_pairs=None, dpo_heldout_pairs=None, sweep_steps=100)
# (None = all pairs: 8,000 train and 1,000 held-out.)

# Hyperparameters. STARTING VALUES, not yet confirmed on a T4: scripts/lab10_seed_protocol.py
# runs seeds 0-4 and these are kept only if the reward-hacking signature holds on every seed.
HP = dict(
    BETA=0.1,  # KL coefficient of Exercise 3, the stretch's center, and DPO's beta
    LR=3e-5,  # AdamW learning rate of rlhf_train (full fine-tuning, no weight decay)
    DPO_LR=5e-6,  # AdamW learning rate of dpo_train
    CLIP=1.0,  # gradient-norm clipping in both loops
)
if OFFLINE == "tiny":  # test-only: a tiny random stand-in barely moves at GPT-2's learning rates
    HP.update(LR=1e-3, DPO_LR=1e-3)
HP.update(json.loads(os.environ.get("NLP_LLMS_LAB10_OVERRIDES", "{}")))  # the seed protocol only
BETA, LR, DPO_LR, CLIP = HP["BETA"], HP["LR"], HP["DPO_LR"], HP["CLIP"]
KL_CHUNK = 64  # responses per chunk for the exact KL and every evaluation

# Thresholds of the trained-policy checkpoints (3, 4, 5 and the stretch). PROVISIONAL TARGETS:
# they must be replaced by the values scripts/lab10_seed_protocol.py prints after seeds 0-4 on a T4
# (each threshold is half the smallest gap seen across the five seeds). Never loosen one to pass.
THRESHOLDS_SOURCE = "provisional targets (2026-10-05), not yet set from the seed protocol"
THRESHOLDS = dict(
    reward_gain_min=0.5,  # Checkpoint 3: mean r_phi(BETA policy) - mean r_phi(pi_ref), normalized units
    drift_max=10.0,  # Checkpoint 3: mean exact KL of the BETA policy, nats per response
    hack_score_tol=0.0,  # Checkpoint 4 (a): r_phi(beta=0) >= r_phi(BETA) - tol
    hack_drift_ratio=3.0,  # Checkpoint 4 (b): drift(beta=0) >= ratio * drift(BETA)
    hack_drift_floor=5.0,  # Checkpoint 4 (b): drift(beta=0) >= floor, nats per response
    hack_gold_margin=0.1,  # Checkpoint 4 (c): gold(BETA) - gold(beta=0) >= margin
    hack_distinct_margin=0.02,  # Checkpoint 4 (d): distinct-2(BETA) - distinct-2(beta=0) >= margin
    dpo_acc_min=0.55,  # Checkpoint 5: held-out accuracy of DPO's implicit reward
    sweep_monotone=True,  # stretch: drift falls as beta rises
)
THRESHOLDS.update(json.loads(os.environ.get("NLP_LLMS_LAB10_THRESHOLDS", "{}")))  # the seed protocol only

CHECK_FAILURES = []  # filled only in the seed protocol


def check(condition, message):
    """A checkpoint about a trained policy. Raises like assert; in the seed protocol it records the
    failure instead, so that every seed's numbers are kept."""
    if condition:
        return True
    if PROTOCOL:
        CHECK_FAILURES.append(message)
        print("PROTOCOL: check would fail:", message)
        return False
    raise AssertionError(message)


NAVY, ACCENT, GREY, GREEN = "#16324f", "#b3541e", "#8a8a8a", "#2e7d4f"  # plot colors
LAB_START = time.time()
TIMES = {}  # stage -> seconds, printed at the end of the core path
print(f"torch {torch.__version__} · transformers {transformers.__version__}")
print(f"device {DEVICE} · FAST {FAST} · seed {SEED} · run sizes {RUN}")
print(f"hyperparameters {HP}")
if FAST:
    print("\nCPU runtime: unit checkpoints run in full; trainings are smoke tests; trained-policy checks are skipped.")
if OFFLINE:
    print(f"\n*** OFFLINE TEST MODE ({OFFLINE}): random stand-in policy, vocabulary, reward model and pairs. ***")

## The Lab 9 Interface, Restated

Notebooks cannot import each other, so this cell restates, unchanged, what Lab 9 defined and Lab 10 uses: the fixed lengths, the **gold rule**, and the functions of Module 9 (returns, REINFORCE loss, baselines, Bradley–Terry).

**The gold rule is a hidden rule we wrote**, not human preference. Lab 9's preference labels were drawn from it through the Bradley–Terry model, which is the only reason this lab can compute a "true" score next to the reward model's. A real RLHF run has no gold reward.

In symbols, $g(x, y) = \min(n_{\text{pos}}, 3) - n_{\text{neg}} - 10 \max(0, f_{\text{list}} - 0.25)$: one point per distinct positive word up to three, minus one per negative word, minus a penalty when more than a quarter of the response's words come from the two lists.

In [ ]:
# Restated from Lab 9 (the "Lab 9 -> Lab 10 interface"). Change both labs or neither.
PROMPT_LEN = 8      # tokens in every prompt; the build script asserts it
RESPONSE_LEN = 24   # tokens in every response; end-of-text is never sampled
EOS_ID = 50256      # == tokenizer.eos_token_id == len(tokenizer) - 1; asserted on load

WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)*")
GOLD = {
    "positive": ["good", "great", "happy", "love", "loved", "wonderful", "beautiful", "nice",
                 "best", "amazing", "perfect", "excellent", "glad", "fun", "enjoyed",
                 "delicious", "proud", "lovely", "fantastic", "excited"],
    "negative": ["bad", "sad", "terrible", "awful", "hate", "angry", "worst", "horrible",
                 "poor", "wrong", "sick", "afraid", "worried", "upset", "disappointed",
                 "pain", "ugly", "cried", "scared", "dead"],
    "cap": 3, "crowd_threshold": 0.25, "crowd_weight": 10.0, "tau_label": 0.5,
}


def complete_words(prompt, response):
    """Lowercased words that end inside the response. A first word that continues the
    prompt's last word is joined to it; a word that touches the end may have been cut
    by the length limit, so it is dropped."""
    text = prompt + response
    return [m.group().lower() for m in WORD.finditer(text) if len(prompt) < m.end() < len(text)]


def gold_reward(prompt, response, gold=GOLD):
    words = complete_words(prompt, response)
    if not words:
        return 0.0
    pos, neg = set(gold["positive"]), set(gold["negative"])
    n_pos = len({w for w in words if w in pos})             # distinct positive words
    n_neg = sum(w in neg for w in words)                    # every negative occurrence
    f_list = sum(w in pos or w in neg for w in words) / len(words)
    return float(min(n_pos, gold["cap"]) - n_neg
                 - gold["crowd_weight"] * max(0.0, f_list - gold["crowd_threshold"]))


def returns_to_go(rewards):
    """G_t = r_t + r_{t+1} + ... + r_T. rewards: (N, T) -> (N, T)."""
    return rewards.flip(-1).cumsum(-1).flip(-1)  # reversed running sum along time


def reinforce_loss(logp, returns, baseline=0.0):
    """-((returns - baseline) * logp).sum(-1).mean(), with the weights treated as constants.

    logp, returns: (N, T). baseline: scalar, (1, T) or (N, T).
    """
    weight = (returns - baseline).detach()  # constants: R is never differentiated
    return -(weight * logp).sum(-1).mean()  # sum over t, mean over samples


def loo_baseline(returns):
    """b_t^(i) = mean of returns[j, t] over the other samples j != i. (N, T) -> (N, T); N >= 2."""
    N = returns.shape[0]
    return (returns.sum(0, keepdim=True) - returns) / (N - 1)  # (1, T) - (N, T): the others' sum


def batch_mean_baseline(returns):
    """(N, T) -> (1, T): per position, the mean over the batch (briefing section 4)."""
    return returns.mean(dim=0, keepdim=True)


def bt_loss(reward_w, reward_l):
    """-log sigmoid(reward_w - reward_l), averaged. reward_w, reward_l: (n,) -> scalar."""
    return -F.logsigmoid(reward_w - reward_l).mean()


def bt_prob(g_a, g_b, tau_label):
    """sigmoid((g_a - g_b) / tau_label): P(a preferred to b). Floats or tensors."""
    p = torch.sigmoid((torch.as_tensor(g_a, dtype=torch.float64) - g_b) / tau_label)
    if isinstance(g_a, (int, float)) and isinstance(g_b, (int, float)):
        return p.item()  # a Python float for float inputs
    return p.to(torch.get_default_dtype())


_x = "My sister opened the old letter and felt"
print(gold_reward(_x, " so happy that she called everyone she knew to tell them the news."))  # 1 positive: 1.0
print(gold_reward(_x, " so happy that she cried."))  # 1 - 1, and 2 of 5 words on the lists: 0 - 10 * 0.15 = -1.5

In [ ]:
# Restated from Lab 9: the reward model class, with Lab 5's attention, mask and block.
# Keep every attribute name: the state_dict keys of data/lab09_reward_model.pt depend on them.
def scaled_dot_product_attention(Q, K, V, mask=None):
    """softmax(Q K^T / sqrt(d_k)) V on the last two dimensions; any leading dimensions."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (..., T, T)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    A = torch.softmax(scores, dim=-1)
    return A @ V, A


def causal_mask(T):
    """(T, T) bool tensor, True where position t may read position i, that is where i <= t."""
    return torch.tril(torch.ones(T, T, dtype=torch.bool))


class MultiHeadAttention(nn.Module):
    """Masked multi-head self-attention (Lab 5)."""

    def __init__(self, d, n_h, dropout=0.0):
        super().__init__()
        assert d % n_h == 0, "d must be divisible by n_h"
        self.n_h, self.d_k = n_h, d // n_h
        self.W_Q = nn.Linear(d, d, bias=False)
        self.W_K = nn.Linear(d, d, bias=False)
        self.W_V = nn.Linear(d, d, bias=False)
        self.proj = nn.Linear(d, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, H, mask):
        B, T, d = H.shape
        Q = self.W_Q(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)  # (B, n_h, T, d_k)
        K = self.W_K(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)
        V = self.W_V(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)
        V_bar, A = scaled_dot_product_attention(Q, K, V, mask)
        V_bar = V_bar.transpose(1, 2).reshape(B, T, d)  # concatenate the heads
        return self.dropout(self.proj(V_bar)), A


class Block(nn.Module):
    """One pre-norm transformer block (Lab 5)."""

    def __init__(self, d, n_h, d_ff, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, n_h, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d), nn.Dropout(dropout))

    def forward(self, H, mask):
        attn_out, A = self.attn(self.ln1(H), mask)
        H = H + attn_out  # H' = H + MHA(LN(H))
        H = H + self.ffn(self.ln2(H))  # H_out = H' + FFN(LN(H'))
        return H, A


class RewardModel(nn.Module):
    """Small transformer trained from scratch on GPT-2 token ids: Lab 5's pre-norm block
    with a causal mask; a scalar head reads the last position of prompt + response."""

    def __init__(self, n_rows, d=64, n_h=4, n_l=2, T_max=32, d_ff=256):
        super().__init__()
        self.T_max = T_max
        # GPT-2 id -> embedding row; row 0 is shared by every id not seen twice in the train pairs.
        self.register_buffer("vocab_map", torch.zeros(EOS_ID + 1, dtype=torch.long))
        self.register_buffer("score_mean", torch.tensor(0.0))  # set by Exercise 5
        self.register_buffer("score_std", torch.tensor(1.0))
        self.tok_emb = nn.Embedding(n_rows, d)
        self.pos_emb = nn.Embedding(T_max, d)
        self.blocks = nn.ModuleList([Block(d, n_h, d_ff, dropout=0.0) for _ in range(n_l)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, 1)
        for module in self.modules():  # GPT-2 style: small normal weights, zero biases (Lab 5)
            if isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def raw(self, prompt_ids, response_ids):
        """(N, P), (N, L) int64 GPT-2 ids -> (N,) raw r_phi."""
        ids = torch.cat([prompt_ids, response_ids], dim=1)  # (N, P + L)
        T = ids.shape[1]
        assert T <= self.T_max, f"prompt + response must be at most {self.T_max} tokens"
        H = self.tok_emb(self.vocab_map[ids]) + self.pos_emb(torch.arange(T, device=ids.device))
        mask = causal_mask(T).to(ids.device)
        for block in self.blocks:
            H, _ = block(H, mask)
        return self.head(self.ln_f(H[:, -1]))[:, 0]  # the last position has read everything

    def forward(self, prompt_ids, response_ids):
        """(N,) normalized: (raw - score_mean) / score_std."""
        return (self.raw(prompt_ids, response_ids) - self.score_mean) / self.score_std

## Data and Model

| | |
|---|---|
| Reference policy $\pi_{\text{ref}}$ | `distilbert/distilgpt2`, the distilled GPT-2 you probed in Lab 6: 6 blocks, 81.9 million parameters, 50,257 tokens, Apache 2.0. Loaded in float32 and always used in `eval()` mode, because its configuration sets dropout to 0.1 |
| Prompts | from Lab 9: 16 subjects × 16 frames, each exactly 8 GPT-2 tokens. 160 train prompts (used for RLHF), 32 held-out, 64 evaluation prompts whose frames occur in no preference pair |
| Preference pairs | from Lab 9: two 24-token continuations sampled from $\pi_{\text{ref}}$, labeled through Bradley–Terry from the gold rule. 8,000 train pairs (the reward model's training data, and DPO's) and 1,000 held-out pairs |
| Reward model $r_\phi$ | from Lab 9: the small transformer above, trained on the train pairs, frozen here. Its output is normalized so that held-out reference responses have mean 0 and standard deviation 1; reward gains below are in these units |

**Sampling rule, used everywhere in this lab:** temperature 1, no top-$k$ or nucleus truncation, from the next-token distribution **without the end-of-text token** (id 50256, GPT-2's last token). So every response has exactly 24 tokens and nothing is ever padded. Log-probabilities, the KL divergence and sampling all use the same function, `policy_logits`, so they describe the same distribution.

The first cell is the workshop's standard loader (`data/README.md` in the repository). It checks each file against a SHA-256 hash, so everyone trains on the same bytes.

In [ ]:
def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


DATA_BASES = [
    "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data",
    "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data",
]
# The three Lab 9 files and their SHA-256. Must match the `datasets` entries lab09_prompts,
# lab09_preferences and lab09_reward_model in _variables.yml (built 2026-10-06 by
# data/build_lab09_preferences.py and Lab 9's own solution code at seed 0).
LAB09_FILES = {
    "prompts": ("lab09_prompts.json", "966971fd2f9f8533711db4cd2f8eaf36e839bd0fb9ed1c1c52efc970f6a25ed6"),
    "preferences": ("lab09_preferences.jsonl.gz", "c00b775865dabf491542719b223ab6fb539f3728ca0348c8ca50ecb51b752326"),
    "reward_model": ("lab09_reward_model.pt", "61c13537e4d8d0fd82e20006033f256937da01746885816c2522d607e6fad183"),
}


def load_lab09(key, folder=None):
    """The bytes of one Lab 9 file. `folder`: a local directory to read instead (test-only stand-in)."""
    name, sha256 = LAB09_FILES[key]
    if folder is not None:
        return (Path(folder) / name).read_bytes()
    urls = [f"{base}/{name}" for base in DATA_BASES]
    if sha256 is not None:
        return fetch(name, urls, sha256)
    print(f"WARNING: {name} has no pinned SHA-256 yet; it is loaded without a hash check.")
    for path in (Path(os.environ.get("NLP_LLMS_DATA", "data")) / name, Path("../data") / name):
        if path.exists():
            return path.read_bytes()
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                return response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
    raise RuntimeError(f"Could not load {name}. Lab 9's data files must be built and committed first "
                       "(data/build_lab09_preferences.py and the Lab 9 notebook).")

In [ ]:
#@title Test-only: offline stand-in for the model and the Lab 9 files (ignore this cell) { display-mode: "form" }
# Used ONLY when NLP_LLMS_OFFLINE_TINY=1 or NLP_LLMS_LAB10_OFFLINE is set (see the setup cell), by the
# repository's tests on a machine that cannot reach huggingface.co. It builds:
#   - a word-level tokenizer with GPT-2's byte-level spaces ("Ġword") and <|endoftext|> as its LAST
#     id, as in GPT-2: "tiny" has a few hundred tokens; "distilgpt2-shape" is padded to 50,257;
#   - a RANDOMLY INITIALIZED GPT-2: "tiny" (2 blocks, width 64) or "distilgpt2-shape" (exactly
#     distilgpt2's shapes, to check memory and CPU run time);
#   - Lab 9's three files in their exact formats: the prompts of Lab 9's brief, synthetic pairs of
#     random words (not samples from the policy), and a RewardModel trained on them for a few seconds.
# The rest of the notebook runs unchanged on them. Nothing measured on this stand-in is a result.


def build_offline_stand_in(kind):
    global EOS_ID
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers
    from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast

    subjects = ["My mother", "My father", "My sister", "My brother", "My friend", "My boss", "Our neighbor",
                "The teacher", "The doctor", "The manager", "The waiter", "The student", "The driver",
                "The coach", "The nurse", "The chef"]
    frames = {
        "train": ["looked at the results feeling so", "opened the letter and felt so", "walked into the room feeling so",
                  "tasted the soup and felt so", "read the review and felt so", "came home late and felt so",
                  "listened to the song feeling so", "looked out the window feeling so", "heard the news and felt so",
                  "finished the long day feeling so"],
        "heldout": ["watched the game and felt so", "visited the old house feeling so"],
        "eval": ["left the new restaurant feeling so", "read the email and felt so", "saw the bill and felt so",
                 "spent the whole weekend feeling so"],
    }
    filler = ("the a and to of it was is that he she they we I you my our his her very so but not with for this "
              "just all really when then there what like at on in as be have had been one day time home back "
              "today again more some much").split()
    lists = GOLD["positive"] + GOLD["negative"]
    prompt_words = sorted({w for s in subjects for w in s.split()} | {w for f in sum(frames.values(), []) for w in f.split()})
    words = list(dict.fromkeys(filler + lists + prompt_words))
    vocab = {w: i for i, w in enumerate(["My", "The", "Our", ",", ".", "!"])}  # no space before these
    for w in words:
        vocab.setdefault("Ġ" + w, len(vocab))  # GPT-2's byte-level space: "Ġword" decodes to " word"
    n_target = 50256 if kind == "distilgpt2-shape" else len(vocab)
    for i in range(n_target - len(vocab)):
        vocab[f"<unused_{i}>"] = len(vocab)
    vocab["<|endoftext|>"] = len(vocab)
    EOS_ID = len(vocab) - 1  # the stand-in's end-of-text id; GPT-2's is 50256

    out = Path(f"lab10-offline-{kind}")
    out.mkdir(exist_ok=True)
    wl = Tokenizer(models.WordLevel(vocab=vocab, unk_token="<|endoftext|>"))
    wl.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    wl.decoder = decoders.ByteLevel()
    tok = PreTrainedTokenizerFast(tokenizer_object=wl, bos_token="<|endoftext|>", eos_token="<|endoftext|>",
                                  unk_token="<|endoftext|>", model_max_length=1024)
    tok.save_pretrained(out)
    shape = (dict(n_embd=768, n_layer=6, n_head=12) if kind == "distilgpt2-shape"
             else dict(n_embd=64, n_layer=2, n_head=2))
    torch.manual_seed(0)  # fixed across seeds, like the real pi_ref and Lab 9 files
    config = GPT2Config(vocab_size=EOS_ID + 1, n_positions=1024, bos_token_id=EOS_ID, eos_token_id=EOS_ID, **shape)
    GPT2LMHeadModel(config).save_pretrained(out)

    # Lab 9's prompts file.
    def encode(text):
        return tok(text, add_special_tokens=False).input_ids

    prompts = {}
    for split, fs in frames.items():
        prompts[split] = []
        for f in fs:
            for s in subjects:
                text = f"{s} {f}"
                ids = encode(text)
                assert len(ids) == PROMPT_LEN and tok.decode(ids) == text, text
                prompts[split].append({"text": text, "ids": ids})
    rng = random.Random(1234)
    word_ids = [vocab["Ġ" + w] for w in filler]
    list_ids = [vocab["Ġ" + w] for w in lists]
    punct_ids = [vocab[p] for p in (",", ".", "!")]

    def random_response():
        ids = []
        for _ in range(RESPONSE_LEN):
            u = rng.random()
            ids.append(rng.choice(list_ids if u < 0.12 else punct_ids if u < 0.22 else word_ids))
        return ids

    def decode(ids):
        return tok.decode(ids, clean_up_tokenization_spaces=False)

    preview_prompt = prompts["heldout"][0]["text"]
    previews = {"one positive word repeated": [vocab["Ġgood"]] * RESPONSE_LEN,
                "an ordinary reply": [vocab["Ġit"], vocab["Ġwas"], vocab["Ġvery"], vocab["Ġgood"]] + [vocab["."]] * 20}
    prompts_doc = {"format": "nlp-llms/lab09-prompts", "version": 1, "model": str(out), "revision": None,
                   "subjects": subjects, "frames": frames, "prompts": prompts,
                   "preview": [{"name": k, "response": decode(v), "response_ids": v} for k, v in previews.items()]}
    (out / "lab09_prompts.json").write_text(json.dumps(prompts_doc))
    assert preview_prompt

    # Lab 9's pairs file: train pairs on train prompts, held-out pairs on held-out prompts.
    rows = []
    for split, n_pairs in (("train", 640), ("heldout", 160)):
        pool = prompts[split]
        for i in range(n_pairs):
            p = pool[i % len(pool)]
            a, b = random_response(), random_response()
            g_a, g_b = gold_reward(p["text"], decode(a)), gold_reward(p["text"], decode(b))
            if rng.random() >= bt_prob(g_a, g_b, GOLD["tau_label"]):
                a, b, g_a, g_b = b, a, g_b, g_a
            rows.append({"prompt": p["text"], "prompt_ids": p["ids"], "chosen": decode(a), "rejected": decode(b),
                         "chosen_ids": a, "rejected_ids": b, "gold_chosen": g_a, "gold_rejected": g_b, "split": split})
    with gzip.open(out / "lab09_preferences.jsonl.gz", "wt", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    # Lab 9's reward model: vocab_map from the train pairs, a few seconds of Bradley-Terry training.
    def as_tensors(split):
        sel = [r for r in rows if r["split"] == split]
        return tuple(torch.tensor([r[k] for r in sel]) for k in ("prompt_ids", "chosen_ids", "rejected_ids"))

    tx, tw, tl = as_tensors("train")
    hx, hw, hl = as_tensors("heldout")
    counts = torch.bincount(torch.cat([tx.flatten(), tw.flatten(), tl.flatten()]), minlength=EOS_ID + 1)
    kept = (counts >= 2).nonzero().squeeze(1)  # ids seen at least twice get their own row, from 1
    rm = RewardModel(n_rows=len(kept) + 1)
    rm.vocab_map[kept] = torch.arange(1, len(kept) + 1)
    opt = torch.optim.Adam(rm.parameters(), lr=3e-3)
    gen = torch.Generator().manual_seed(0)
    for _ in range(200):
        idx = torch.randint(len(tx), (64,), generator=gen)
        loss = bt_loss(rm.raw(tx[idx], tw[idx]), rm.raw(tx[idx], tl[idx]))
        opt.zero_grad()
        loss.backward()
        opt.step()
    rm.eval()
    with torch.no_grad():
        raw_w, raw_l = rm.raw(hx, hw), rm.raw(hx, hl)
        both = torch.cat([raw_w, raw_l])
        rm.score_mean.fill_(both.mean())
        rm.score_std.fill_(both.std(unbiased=False))
    g_w = torch.tensor([r["gold_chosen"] for r in rows if r["split"] == "heldout"])
    g_l = torch.tensor([r["gold_rejected"] for r in rows if r["split"] == "heldout"])
    untied = g_w != g_l
    ranks = lambda v: torch.argsort(torch.argsort(v)).float()  # noqa: E731
    spearman = torch.corrcoef(torch.stack([ranks(both), ranks(torch.cat([g_w, g_l]))]))[0, 1]
    ckpt = {"format": "nlp-llms/lab09-reward-model", "version": 2,
            "config": {"n_rows": len(kept) + 1, "d": 64, "n_h": 4, "n_l": 2, "T_max": 32, "d_ff": 256},
            "policy": {"model": str(out), "revision": None, "eos_id": EOS_ID, "vocab_size": EOS_ID + 1,
                       "prompt_len": PROMPT_LEN, "response_len": RESPONSE_LEN},
            "state_dict": rm.state_dict(), "gold": GOLD,
            "metrics": {"heldout_acc": float((raw_w > raw_l).float().mean()),
                        "oracle_acc": float(((g_w > g_l).float() + 0.5 * (g_w == g_l).float()).mean()),
                        "acc_gold_order_untied": float(((raw_w > raw_l) == (g_w > g_l))[untied].float().mean()),
                        "spearman_gold": float(spearman)},
            "data_sha256": {key: hashlib.sha256((out / LAB09_FILES[key][0]).read_bytes()).hexdigest()
                            for key in ("prompts", "preferences")}}
    torch.save(ckpt, out / "lab09_reward_model.pt")
    return str(out), str(out)

In [ ]:
# Must match models.causal_lm and models.causal_lm_revision in _variables.yml.
CAUSAL_LM = "distilbert/distilgpt2"
CAUSAL_LM_REVISION = "2290a62682d06624634c1f46a6ad5be0f47f38aa"  # the commit Lab 9's pairs were sampled from

started = time.time()
print("Loading the model and Lab 9's files; the first run downloads about 350 MB ...")
if OFFLINE:
    MODEL_SOURCE, LAB09_DIR = build_offline_stand_in(OFFLINE)  # test-only: see the hidden cell above
    LOAD_KWARGS = {}
else:
    MODEL_SOURCE, LAB09_DIR = CAUSAL_LM, None
    LOAD_KWARGS = {"revision": CAUSAL_LM_REVISION} if CAUSAL_LM_REVISION else {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, **LOAD_KWARGS)
pi_ref = AutoModelForCausalLM.from_pretrained(MODEL_SOURCE, dtype=torch.float32, **LOAD_KWARGS)
pi_ref = pi_ref.to(DEVICE).eval().requires_grad_(False)  # frozen; eval() turns dropout off
assert tokenizer.eos_token_id == EOS_ID == len(tokenizer) - 1, (tokenizer.eos_token_id, EOS_ID, len(tokenizer))
assert pi_ref.config.vocab_size == EOS_ID + 1

# Lab 9's prompts.
prompts_blob = load_lab09("prompts", LAB09_DIR)
prompts_doc = json.loads(prompts_blob)
assert prompts_doc["format"] == "nlp-llms/lab09-prompts" and prompts_doc["version"] == 1
assert prompts_doc["model"] == (CAUSAL_LM if not OFFLINE else MODEL_SOURCE), prompts_doc["model"]
PROMPT_TEXT = {}  # token ids -> text, for the gold rule
for split, rows in prompts_doc["prompts"].items():
    for p in rows:
        assert len(p["ids"]) == PROMPT_LEN and EOS_ID not in p["ids"], p
        PROMPT_TEXT[tuple(p["ids"])] = p["text"]
train_prompt_ids = torch.tensor([p["ids"] for p in prompts_doc["prompts"]["train"]], device=DEVICE)  # (160, 8)
eval_prompt_ids = torch.tensor([p["ids"] for p in prompts_doc["prompts"]["eval"]], device=DEVICE)  # (64, 8)

# Lab 9's preference pairs.
pairs_blob = load_lab09("preferences", LAB09_DIR)
pairs = [json.loads(line) for line in gzip.decompress(pairs_blob).decode("utf-8").splitlines() if line.strip()]


def pair_tensors(split, limit=None):
    """(prompt_ids (n, 8), chosen_ids (n, 24), rejected_ids (n, 24), gold_chosen (n,), gold_rejected (n,))."""
    rows = [r for r in pairs if r["split"] == split][:limit]
    ids = [torch.tensor([r[k] for r in rows], device=DEVICE) for k in ("prompt_ids", "chosen_ids", "rejected_ids")]
    gold = [torch.tensor([r[k] for r in rows]) for k in ("gold_chosen", "gold_rejected")]
    return (*ids, *gold)


train_pairs = pair_tensors("train", RUN["dpo_train_pairs"])
heldout_pairs = pair_tensors("heldout", RUN["dpo_heldout_pairs"])
for t in (*train_pairs[:3], *heldout_pairs[:3]):
    assert not (t == EOS_ID).any()
assert train_pairs[1].shape[1] == heldout_pairs[2].shape[1] == RESPONSE_LEN

# Lab 9's reward model, frozen.
rm_path = os.environ.get("NLP_LLMS_LAB10_REWARD_MODEL")  # the seed protocol's robustness check only
rm_blob = Path(rm_path).read_bytes() if rm_path else load_lab09("reward_model", LAB09_DIR)
ckpt = torch.load(io.BytesIO(rm_blob), weights_only=True)
assert ckpt["format"] == "nlp-llms/lab09-reward-model" and ckpt["version"] == 2
assert ckpt["policy"]["model"] == prompts_doc["model"], ckpt["policy"]
assert (ckpt["policy"]["eos_id"], ckpt["policy"]["vocab_size"]) == (EOS_ID, len(tokenizer)), ckpt["policy"]
assert (ckpt["policy"]["prompt_len"], ckpt["policy"]["response_len"]) == (PROMPT_LEN, RESPONSE_LEN)
if CAUSAL_LM_REVISION is not None and not OFFLINE:  # the test-only stand-in has no revision
    assert ckpt["policy"]["revision"] == CAUSAL_LM_REVISION == prompts_doc["revision"], ckpt["policy"]
assert ckpt["gold"] == GOLD, "the reward model was trained with a different gold rule: update GOLD above"
for key, blob in (("prompts", prompts_blob), ("preferences", pairs_blob)):
    assert ckpt["data_sha256"][key] == hashlib.sha256(blob).hexdigest(), f"the reward model was trained on another {key} file"
reward_model = RewardModel(**ckpt["config"])
reward_model.load_state_dict(ckpt["state_dict"])
reward_model = reward_model.to(DEVICE).eval().requires_grad_(False)

TIMES["load model and data"] = time.time() - started
n_params = sum(p.numel() for p in pi_ref.parameters())
print(f"{type(pi_ref).__name__}: {n_params:,} parameters, {pi_ref.config.n_layer} blocks · "
      f"vocabulary {len(tokenizer):,} · end-of-text id {EOS_ID}{NOTE}")
print(f"prompts: {len(train_prompt_ids)} train, {len(eval_prompt_ids)} evaluation · pairs: "
      f"{len(train_pairs[0]):,} train, {len(heldout_pairs[0]):,} held-out (of {len(pairs):,})")
print(f"reward model: {sum(p.numel() for p in reward_model.parameters()):,} parameters · Lab 9 metrics "
      f"{ {k: round(v, 3) for k, v in ckpt['metrics'].items()} }")
print(f"({TIMES['load model and data']:.1f} s)")

**Provided helpers.** Read the docstrings; you will call `policy_logits` in Exercise 1.

- `policy_logits(model, input_ids, cache=None)`: the model's next-token logits with the end-of-text column removed, shape `(N, T, 50256)`. Logit $t$ is the model's score for the token at position $t + 1$.
- `sample_responses(policy, prompt_ids, generator)`: 24 tokens per prompt at temperature 1, built on `policy_logits` with a **KV cache** (each step feeds only the new token; the keys and values of earlier positions are kept in `cache`). A `torch.Generator` makes every sample reproducible.
- `score(prompt_ids, response_ids)`: $r_\phi(x, y)$. `gold_of(prompt_ids, response_ids)`: decodes the response and applies the gold rule.
- `distinct_n(token_lists, n)`: distinct token n-grams divided by all token n-grams, pooled over a list of responses.

In [ ]:
def policy_logits(model, input_ids, cache=None):
    """Next-token logits without the end-of-text column: (N, T) ids -> (N, T, EOS_ID).
    With a DynamicCache, input_ids holds only the new positions and the cache is updated in place."""
    out = model(input_ids=input_ids, past_key_values=cache, use_cache=cache is not None)
    return out.logits[..., :EOS_ID]  # GPT-2's end-of-text id is its last row


def make_generator(seed):
    return torch.Generator(device=DEVICE).manual_seed(seed)


@torch.no_grad()
def sample_responses(policy, prompt_ids, generator, length=RESPONSE_LEN):
    """Sample `length` tokens per prompt: temperature 1, no truncation, no end-of-text. (N, P) -> (N, length)."""
    cache = DynamicCache(config=policy.config)
    logits = policy_logits(policy, prompt_ids, cache)[:, -1]  # (N, EOS_ID): the first response token
    tokens = []
    for t in range(length):
        probs = torch.softmax(logits.float(), dim=-1)
        nxt = torch.multinomial(probs, 1, generator=generator)  # (N, 1)
        tokens.append(nxt)
        if t + 1 < length:
            logits = policy_logits(policy, nxt, cache)[:, -1]  # one new position, keys and values cached
    return torch.cat(tokens, dim=1)


@torch.no_grad()
def sample_many(policy, prompt_ids, generator, chunk=128):
    """sample_responses in chunks, drawing from one generator in order (same result for any chunk size of the same order)."""
    return torch.cat([sample_responses(policy, prompt_ids[i : i + chunk], generator)
                      for i in range(0, len(prompt_ids), chunk)])


@torch.no_grad()
def score(prompt_ids, response_ids, chunk=256):
    """r_phi(x, y), normalized, for every row: (N,) on the CPU."""
    return torch.cat([reward_model(prompt_ids[i : i + chunk], response_ids[i : i + chunk]).cpu()
                      for i in range(0, len(prompt_ids), chunk)])


def decode(response_ids):
    return tokenizer.batch_decode(response_ids.tolist(), clean_up_tokenization_spaces=False)


def gold_of(prompt_ids, response_ids):
    """The gold rule's score for every row: (N,). Known only because the data are synthetic."""
    prompts = [PROMPT_TEXT[tuple(p)] for p in prompt_ids.tolist()]
    return torch.tensor([gold_reward(p, y) for p, y in zip(prompts, decode(response_ids))])


def distinct_n(token_lists, n=2):
    """Distinct token n-grams / all token n-grams, pooled over the given responses."""
    grams = [tuple(t[i : i + n]) for t in token_lists for i in range(len(t) - n + 1)]
    return len(set(grams)) / max(len(grams), 1)


def show(text, width=100, indent="    "):
    print(textwrap.indent(textwrap.fill(text.replace("\n", " ⏎ "), width), indent))


assert distinct_n([[1, 2, 3], [1, 2, 4]]) == 3 / 4  # bigrams (1,2) (2,3) (1,2) (2,4)

# Step 0 · The Reward Model and the Gold Rule on the Reference's Samples (3 minutes)

The cell samples four responses from $\pi_{\text{ref}}$ for each of three evaluation prompts, and prints the reward model's score $r_\phi(x, y)$ (normalized: 0 is an average reference response) next to the gold reward $g(x, y)$.

**Predict.** On text like this, the reward model's training data, do you expect $r_\phi$ and $g$ to rank the responses in the same order? Where might they disagree?

In [ ]:
started = time.time()
_gen = make_generator(EVAL_SEED)
_prompts = eval_prompt_ids[[0, 16, 32]].repeat_interleave(4, dim=0)  # 3 prompts with different frames, 4 samples each
_responses = sample_responses(pi_ref, _prompts, _gen)
_scores, _golds = score(_prompts, _responses), gold_of(_prompts, _responses)
for i, text in enumerate(decode(_responses)):
    if i % 4 == 0:
        print(f"PROMPT: {PROMPT_TEXT[tuple(_prompts[i].tolist())]!r}")
    print(f"  r_phi {_scores[i]:+.2f} · gold {_golds[i]:+.2f} · {text!r}")
_rank = lambda v: v.argsort().argsort().numpy()  # noqa: E731
print(f"Spearman correlation over these 12: {np.corrcoef(_rank(_scores), _rank(_golds))[0, 1]:.2f}"
      f" (ties broken arbitrarily){NOTE}")
TIMES["step 0"] = time.time() - started

**Explain.** On the reference's own samples the two mostly agree: Lab 9 checked that the reward model recovers the gold ordering on held-out reference samples. Responses with no list word score near 0 on both. Disagreements here are the reward model's own errors, for instance on words it saw rarely. The gold rule also has two parts that the reference's samples almost never trigger, the cap at three positive words and the penalty for crowding, so the reward model never learned them. Lab 9 set them up that way on purpose. Remember this when you remove the KL penalty in Exercise 4.

# Part A · The Pieces of the Objective

## Exercise 1 · Log-Probabilities of a Response (8 minutes)

The policy gradient, the KL penalty and DPO all need $\log \pi(y_t \mid s_t)$ for every response token, where $s_t = (x, y_{<t})$ is the prompt and the response so far. Their sum is the sequence log-probability (briefing, eq. `seq-logprob`):

$$
\log \pi_\theta(y \mid x) = \sum_{t=1}^{|y|} \log \pi_\theta(y_t \mid s_t).
$$

Concatenate prompt and response into one sequence of length $P + L$ and run the model once. Logit position $j$ scores the token at position $j + 1$, so the logits that score $y_1, \dots, y_L$ sit at positions $P - 1, \dots, P + L - 2$.

**Predict.** For 2 prompts of 8 tokens and responses of 24 tokens, what shape does `policy_logits` return, and which slice of it do you need? What should the log-ratio $\log \pi_\theta(y_t \mid s_t) - \log \pi_{\text{ref}}(y_t \mid s_t)$ be for a policy that is an exact copy of the reference?

**Task.** Complete `response_logprobs(model, prompt_ids, response_ids)`: call `policy_logits` on the concatenation, keep the $L$ positions that predict the response, take `torch.log_softmax` over the vocabulary, and pick each response token's entry with `gather`. Return shape `(N, L)`.

<details>
<summary>Hint for TODO 1</summary>

Logit position $j$ scores the token at position $j + 1$, so $y_1$ is scored at position $P - 1$ and $y_L$ at $P + L - 2$: keep the $L$ positions from $P - 1$ on and drop the last one (briefing eq. `seq-logprob`). A slice that starts at $P$ is off by one token, which the checkpoint's position-by-position comparison shows as an error of several nats. Take the log-softmax over the vocabulary, then gather each response token's entry.

</details>

In [ ]:
# TODO 1: complete this function.
def response_logprobs(model, prompt_ids, response_ids):
    """log pi(y_t | s_t) for every response position: (N, P), (N, L) -> (N, L)."""
    P = prompt_ids.shape[1]
    input_ids = torch.cat([prompt_ids, response_ids], dim=1)  # (N, P + L)
    ...

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def response_logprobs(model, prompt_ids, response_ids):
    """log pi(y_t | s_t) for every response position: (N, P), (N, L) -> (N, L)."""
    P = prompt_ids.shape[1]
    input_ids = torch.cat([prompt_ids, response_ids], dim=1)  # (N, P + L)
    logits = policy_logits(model, input_ids)[:, P - 1 : -1]  # (N, L, V): position P-1+t scores y_t
    logp_all = torch.log_softmax(logits.float(), dim=-1)  # (N, L, V)
    return logp_all.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)  # (N, L)

<details>
<summary>Why this solution works</summary>

A causal model's output at position $j$ depends only on tokens $1, \dots, j$, so one forward pass over prompt plus response gives the next-token distribution at every state $s_t$ at once. The slice `[:, P - 1 : -1]` drops the positions inside the prompt (they predict prompt tokens, which the policy did not choose) and the last position (it predicts a token after the response). `log_softmax` over the 50,256 columns normalizes without the end-of-text token, the same distribution the sampler draws from, and `gather` picks $\log \pi(y_t \mid s_t)$ for the token actually sampled. Summing over the last dimension gives eq. `seq-logprob`.

</details>

In [ ]:
# Checkpoint 1
workshop.checkpoint(1)
_p = eval_prompt_ids[:2]  # (2, 8)
_y = sample_responses(pi_ref, _p, make_generator(EVAL_SEED), length=5)  # (2, 5)
with torch.no_grad():
    _logp = response_logprobs(pi_ref, _p, _y)
    # The same numbers position by position, one forward pass per prefix (provided).
    _hand = torch.zeros(2, 5, device=DEVICE)
    for i in range(2):
        for t in range(5):
            _prefix = torch.cat([_p[i], _y[i, :t]])[None]  # (1, P + t): the state s_t
            _hand[i, t] = torch.log_softmax(policy_logits(pi_ref, _prefix)[0, -1], dim=-1)[_y[i, t]]
assert _logp.shape == (2, 5), f"expected (N, L) = (2, 5), got {tuple(_logp.shape)}: drop the prompt positions"
_err = (_logp - _hand).abs().max().item()
print(f"max |response_logprobs - position-by-position| = {_err:.2e}")
# 1e-4: one batched pass and one pass per prefix differ by float32 rounding alone, up to 3.0e-5 on a
# CPU (measured 2026-10-06 over 32 prompt pairs); a slice off by one position is off by about 11 nats.
assert _err < 1e-4, "your log-probabilities do not match: check the slice (positions P-1 to P+L-2)"

# Before training, a copy of the reference has log-ratio exactly zero.
policy = copy.deepcopy(pi_ref)  # both float32, eval(), on DEVICE
_y24 = sample_responses(pi_ref, eval_prompt_ids[:4], make_generator(EVAL_SEED))
with torch.no_grad():
    _ratio = response_logprobs(policy, eval_prompt_ids[:4], _y24) - response_logprobs(pi_ref, eval_prompt_ids[:4], _y24)
with torch.no_grad():
    _seq = response_logprobs(pi_ref, eval_prompt_ids[:4], _y24).sum(1)  # eq. seq-logprob
print(f"log pi_ref(y | x) of 4 sampled responses: {[round(v, 1) for v in _seq.tolist()]} nats{NOTE}")
if torch.equal(_ratio, torch.zeros_like(_ratio)):
    print("log-ratio of the copy to the reference: exactly 0")
else:  # flag 5 of the brief: should not happen with the deterministic settings of the setup cell
    print(f"WARNING: log-ratio not bit-exact on this device (max {_ratio.abs().max().item():.1e}); using atol 1e-6")
    assert _ratio.abs().max() < 1e-6
del policy
print(f"Checkpoint 1 passed{NOTE}")

## Exercise 2 · The Per-Token Reward With a KL Penalty (8 minutes)

The objective of RLHF (briefing, eq. `rlhf-objective`) is the reward minus $\beta$ times the drift from the reference. Split over tokens (eq. `token-reward`):

$$
r_t = -\beta\,\big(\log \pi_\theta(y_t \mid s_t) - \log \pi_{\text{ref}}(y_t \mid s_t)\big) \;+\; \mathbb{1}[t = |y|]\; r_\phi(x, y).
$$

Every token pays for its own departure from the reference; the reward model's score arrives at the last token. The log-ratio is computed, then frozen: no gradient flows through the reward.

**Predict.** With $\beta = 0$, which entries of the `(N, L)` reward tensor are nonzero? What is the sum of a row for general $\beta$?

**Task.** Complete `token_rewards(score, logp, logp_ref, beta)`: `score` is `(N,)`, `logp` and `logp_ref` are `(N, L)`. Return the `(N, L)` rewards. Call `.detach()` on the log-ratio.

<details>
<summary>Hint for TODO 2</summary>

Each token's reward is minus $\beta$ times its own log-ratio, $\log \pi_\theta - \log \pi_{\text{ref}}$ (briefing eq. `token-reward`): a token the policy now likes more than the reference costs reward, and with the sign flipped the policy would be paid to drift. Add the reward model's score to the last column only, and detach the log-ratio: the rewards are constants for the policy gradient.

</details>

In [ ]:
# TODO 2: complete this function.
def token_rewards(score, logp, logp_ref, beta):
    """score: (N,) reward-model scores; logp, logp_ref: (N, L) log-probs of the sampled tokens."""
    raise NotImplementedError("TODO 2")

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def token_rewards(score, logp, logp_ref, beta):
    """score: (N,) reward-model scores; logp, logp_ref: (N, L) log-probs of the sampled tokens."""
    rewards = -beta * (logp - logp_ref).detach()   # every token pays for its drift
    rewards[:, -1] += score                        # r_phi arrives at the last token
    return rewards                                 # (N, L)

<details>
<summary>Why this solution works</summary>

Summed over $t$, the rewards give $r_\phi(x, y) - \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$, the quantity inside the expectation of the objective, because the sequence log-ratio is the sum of the per-token log-ratios (eq. `seq-logprob`). `.detach()` makes the penalty a fixed number for the gradient: the policy gradient differentiates only $\log \pi_\theta$ in the loss, and the term that differentiating the penalty itself would add has expectation zero (briefing, section 4). All responses have the same length, so the score always goes in the last column. `rewards` is a new tensor, so `+=` does not modify `logp`.

</details>

In [ ]:
# Checkpoint 2
workshop.checkpoint(2)
_score = torch.tensor([1.0, -2.0])
_logp = torch.tensor([[-1.0, -2.0, -3.0], [-1.0, -1.0, -1.0]], requires_grad=True)
_logp_ref = torch.tensor([[-2.0, -2.0, -1.0], [-1.0, -3.0, -1.0]])
_r0 = token_rewards(_score, _logp, _logp_ref, beta=0.0)
assert _r0.shape == (2, 3), f"the rewards have the shape of logp, (N, L) = (2, 3); got {tuple(_r0.shape)}"
assert torch.equal(_r0, torch.tensor([[0.0, 0.0, 1.0], [0.0, 0.0, -2.0]])), f"beta = 0: only the score, at the end\n{_r0}"
_r = token_rewards(_score, _logp, _logp_ref, beta=0.5)
_expected = _score - 0.5 * (_logp - _logp_ref).sum(1)
assert torch.allclose(_r.sum(1), _expected), (
    f"row sums {_r.sum(1).tolist()}, expected score - beta * sum of log-ratios = {_expected.tolist()}: "
    "each token's reward is -beta * (logp - logp_ref); check the sign")
assert torch.equal(_r[:, :-1], -0.5 * (_logp - _logp_ref).detach()[:, :-1]), "each token pays -beta * its log-ratio"
assert not _r.requires_grad, "no gradient may flow through the rewards: detach the log-ratio"
print("rewards with beta = 0.5:\n", _r)
print("Checkpoint 2 passed")

## Exercise 3 · Exact KL, Then RLHF With a KL Penalty (11 minutes)

The penalty in `token_rewards` uses the **sampled** log-ratio, which can be negative for one token. To **measure** drift we use the exact divergence, summed over response positions (eqs. `kl` and `kl-chain`):

$$
\mathrm{KL}\big(\pi_\theta(\cdot \mid x) \,\Vert\, \pi_{\text{ref}}(\cdot \mid x)\big) = \mathbb{E}_{y \sim \pi_\theta}\Big[\sum_{t=1}^{|y|} \sum_{w \in V} \pi_\theta(w \mid s_t) \log \frac{\pi_\theta(w \mid s_t)}{\pi_{\text{ref}}(w \mid s_t)}\Big].
$$

For each response we compute the inner sums exactly, over all 50,256 tokens at each of the 24 positions, and average over responses.

**Predict.** For two identical sets of logits, what is the KL? Can it ever be negative?

**Task.** Complete `exact_kl(logits, logits_ref)`. Both inputs have shape `(N, L, V)`; return `(N,)`: the per-position KL over the last dimension, summed over positions. Use `log_softmax` for both, not `softmax` followed by `log`.

<details>
<summary>Hint for TODO 3</summary>

Weight the log-ratio by the policy's own probabilities, the exponential of its log-softmax, not by $\pi_{\text{ref}}$'s: $\mathrm{KL}(\pi \Vert \pi_{\text{ref}})$ and $\mathrm{KL}(\pi_{\text{ref}} \Vert \pi)$ differ (briefing eq. `kl`). Sum over the vocabulary, the last axis, then over the positions. Take the log-softmax of both sets of logits: a softmax followed by a log gives $\log 0 = -\infty$ wherever a probability underflows.

</details>

In [ ]:
# TODO 3: complete this function.
def exact_kl(logits, logits_ref):
    """sum_t KL(pi(.|s_t) || pi_ref(.|s_t)) for each response: (N, L, V), (N, L, V) -> (N,)."""
    raise NotImplementedError("TODO 3")

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def exact_kl(logits, logits_ref):
    """sum_t KL(pi(.|s_t) || pi_ref(.|s_t)) for each response: (N, L, V), (N, L, V) -> (N,)."""
    logp = torch.log_softmax(logits.float(), dim=-1)  # log pi(w | s_t), (N, L, V)
    logp_ref = torch.log_softmax(logits_ref.float(), dim=-1)
    return (logp.exp() * (logp - logp_ref)).sum(-1).sum(-1)  # over the vocabulary, then the positions

<details>
<summary>Why this solution works</summary>

`(logp.exp() * (logp - logp_ref)).sum(-1)` is eq. `kl` at one position: the expectation, under the policy's next-token distribution, of the log-ratio. Summing over positions gives eq. `kl-chain` for the states the sampled response visited; averaging over sampled responses estimates the expectation over $y$. Working in log space avoids $\log 0$: `log_softmax` subtracts the log-sum-exp, which is stable for any logits. Identical logits give `logp - logp_ref` exactly zero, so the KL is exactly 0.

</details>

In [ ]:
# Checkpoint 3 (unit)
workshop.checkpoint(3)
torch.manual_seed(SEED)
_a, _b = torch.randn(3, 4, 10), torch.randn(3, 4, 10)
_kl = exact_kl(_a, _b)
assert _kl.shape == (3,), f"expected (N,) = (3,), got {tuple(_kl.shape)}"
assert (_kl >= 0).all(), (
    f"a KL divergence is never negative, got {[round(v, 4) for v in _kl.tolist()]}: "
    "weight the log-ratio by the policy's own probabilities")
assert torch.equal(exact_kl(_a, _a), torch.zeros(3)), "identical logits must give exactly 0"
_ref = F.kl_div(torch.log_softmax(_b, -1), torch.log_softmax(_a, -1), log_target=True, reduction="none").sum((-1, -2))
assert torch.allclose(_kl, _ref, atol=1e-5), (
    f"got {[round(v, 4) for v in _kl.tolist()]}, F.kl_div gives {[round(v, 4) for v in _ref.tolist()]}: "
    "check the direction KL(pi || pi_ref), and sum over the vocabulary, then over the positions")
# On real logits: 4 sampled responses, the reference against itself at temperature 2.
with torch.no_grad():
    _lg = policy_logits(pi_ref, torch.cat([eval_prompt_ids[:4], _y24], 1))[:, PROMPT_LEN - 1 : -1]  # (4, 24, V)
    _kl_real = exact_kl(_lg / 2, _lg)
    _ref_real = F.kl_div(torch.log_softmax(_lg, -1), torch.log_softmax(_lg / 2, -1), log_target=True,
                         reduction="none").sum((-1, -2))
print(f"logits of 4 responses: {tuple(_lg.shape)} · KL of the temperature-2 policy from the reference: "
      f"{[round(v, 2) for v in _kl_real.tolist()]} nats{NOTE}")
assert (_kl_real >= 0).all() and torch.equal(exact_kl(_lg, _lg), torch.zeros(4, device=DEVICE)), (
    f"on real logits the KL must be non-negative, and 0 for identical logits; got {_kl_real.tolist()}: "
    "take the log-softmax of both, not a softmax followed by a log")
assert torch.allclose(_kl_real, _ref_real, atol=1e-5 * max(1.0, _ref_real.abs().max().item()))
del _lg
print("Checkpoint 3 (unit) passed")

**Run: RLHF with a KL penalty.** The provided `rlhf_train` puts your three functions together. Each step:

1. draws `batch` train prompts (with replacement) and samples one response for each from the current policy;
2. scores them with the frozen reward model, and computes `logp` (with gradient) and `logp_ref` (without) with your `response_logprobs`;
3. turns them into rewards with your `token_rewards`, then returns $G_t$ and the REINFORCE loss with a per-position batch-mean baseline (eq. `pg-kl`, functions restated from Lab 9);
4. takes one AdamW step on all of the policy's parameters (full fine-tuning; the reference stays frozen).

Every `log_every` steps it records the batch's mean $r_\phi$, mean gold reward, and mean exact KL (your `exact_kl`, in chunks, without gradient). `evaluate(policy)` then measures a policy on 64 evaluation prompts × 8 samples, all drawn with one fixed generator: mean $r_\phi$, mean gold, mean drift (exact KL, nats per response) and distinct-2 (distinct token bigrams over all bigrams, pooled over the 8 samples of each prompt, then averaged over prompts).

One step of REINFORCE per batch, with a batch average as baseline, is the special case of PPO in which the clip never acts (briefing, section 5).

**Predict.** With $\beta$ = `BETA` (printed by the setup cell), will the mean reward-model score on the evaluation prompts rise by a lot, a little, or not at all? Will the drift be closer to 1 nat per response or to 100?

In [ ]:
@torch.no_grad()
def drift(policy, prompt_ids, response_ids, chunk=KL_CHUNK):
    """Exact KL from pi_ref (your exact_kl) for every response, in chunks: (N,) on the CPU."""
    out = []
    for i in range(0, len(prompt_ids), chunk):
        ids = torch.cat([prompt_ids[i : i + chunk], response_ids[i : i + chunk]], dim=1)
        logits = policy_logits(policy, ids)[:, PROMPT_LEN - 1 : -1]  # (n, L, V)
        logits_ref = policy_logits(pi_ref, ids)[:, PROMPT_LEN - 1 : -1]
        out.append(exact_kl(logits, logits_ref).cpu())
        del logits, logits_ref  # free about 0.6 GB per chunk on a GPU
    return torch.cat(out)


@torch.no_grad()
def evaluate(policy):
    """Mean r_phi, gold, exact drift and distinct-2 on the evaluation prompts, one fixed generator."""
    n, k = RUN["eval_prompts"], RUN["eval_samples"]
    prompts = eval_prompt_ids[:n].repeat_interleave(k, dim=0)  # (n * k, P): each prompt k times
    responses = sample_many(policy, prompts, make_generator(EVAL_SEED))
    kl = torch.zeros(len(prompts)) if policy is pi_ref else drift(policy, prompts, responses)
    per_prompt = responses.view(n, k, -1).tolist()
    return {"score": score(prompts, responses).mean().item(), "gold": gold_of(prompts, responses).mean().item(),
            "drift": kl.mean().item(), "distinct2": float(np.mean([distinct_n(s, 2) for s in per_prompt])),
            "responses": responses.cpu()}


def rlhf_train(beta, seed=SEED, steps=None, lr=None, batch=None, log_every=None, verbose=True):
    """KL-penalized REINFORCE from a fresh copy of pi_ref. Returns (policy, log)."""
    steps, lr = steps or RUN["steps"], lr or LR
    batch, log_every = batch or RUN["batch"], log_every or RUN["log_every"]
    torch.manual_seed(seed)
    gen = make_generator(seed)  # prompt draws and samples: the same for every beta with this seed
    policy = copy.deepcopy(pi_ref).requires_grad_(True)  # stays in eval(): no dropout, gradients flow
    opt = torch.optim.AdamW(policy.parameters(), lr=lr, weight_decay=0.0)
    log = {"step": [], "score": [], "gold": [], "kl_exact": [], "kl_sampled": [], "loss": []}
    started = time.time()
    for step in range(steps):
        prompts = train_prompt_ids[torch.randint(len(train_prompt_ids), (batch,), generator=gen, device=DEVICE)]
        responses = sample_responses(policy, prompts, gen)  # (batch, L)
        with torch.no_grad():
            scores = reward_model(prompts, responses)  # (batch,)
            logp_ref = response_logprobs(pi_ref, prompts, responses)  # (batch, L)
        logp = response_logprobs(policy, prompts, responses)  # (batch, L), with gradient
        rewards = token_rewards(scores, logp, logp_ref, beta)
        returns = returns_to_go(rewards)
        loss = reinforce_loss(logp, returns, batch_mean_baseline(returns))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), CLIP)
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            log["step"].append(step)
            log["score"].append(scores.mean().item())
            log["gold"].append(gold_of(prompts, responses).mean().item())
            log["kl_exact"].append(drift(policy, prompts, responses).mean().item())
            log["kl_sampled"].append((logp - logp_ref).sum(1).mean().item())
            log["loss"].append(loss.item())
            if verbose:
                print(f"step {step:>4}  r_phi {log['score'][-1]:+.3f}  gold {log['gold'][-1]:+.3f}  "
                      f"exact KL {log['kl_exact'][-1]:7.3f}  ({time.time() - started:.0f} s)")
    policy.eval().requires_grad_(False)
    return policy, log


def plot_training(logs, title):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    for (name, log), color in zip(logs.items(), (NAVY, ACCENT, GREEN)):
        for ax, key in zip(axes, ("score", "gold", "kl_exact")):
            ax.plot(log["step"], log[key], "o-", ms=3, color=color, label=name)
    for ax, label in zip(axes, ("batch mean r_phi (proxy)", "batch mean gold reward", "batch mean exact KL (nats)")):
        ax.set_xlabel("step")
        ax.set_ylabel(label)
    axes[0].legend(frameon=False)
    fig.suptitle(title + (" [TEST-ONLY stand-in]" if OFFLINE else ""))
    fig.tight_layout()
    plt.show()


started = time.time()
EVAL = {"reference": evaluate(pi_ref)}
TIMES["evaluate pi_ref"] = time.time() - started
print(f"pi_ref on {RUN['eval_prompts']} x {RUN['eval_samples']} responses: r_phi {EVAL['reference']['score']:+.3f} · "
      f"gold {EVAL['reference']['gold']:+.3f} · distinct-2 {EVAL['reference']['distinct2']:.3f}{NOTE}\n")

started = time.time()
policy_beta, log_beta = rlhf_train(beta=BETA)
TIMES["RLHF, beta = BETA"] = time.time() - started
EVAL["rlhf_beta"] = evaluate(policy_beta)
print(f"\n{RUN['steps']} steps of batch {RUN['batch']} in {TIMES['RLHF, beta = BETA']:.1f} s{NOTE}")
plot_training({f"beta = {BETA}": log_beta}, "RLHF with a KL penalty")

In [ ]:
# Checkpoint 3 (after training): on the evaluation prompts, against pi_ref
workshop.checkpoint(3, label="3 (after training)")
gain = EVAL["rlhf_beta"]["score"] - EVAL["reference"]["score"]
print(f"{'':<14}{'r_phi':>8}{'gold':>8}{'drift (nats)':>14}{'distinct-2':>12}")
for name in ("reference", "rlhf_beta"):
    e = EVAL[name]
    print(f"{name:<14}{e['score']:>+8.3f}{e['gold']:>+8.3f}{e['drift']:>14.3f}{e['distinct2']:>12.3f}")
print(f"reward gain {gain:+.3f} (normalized units) · drift {EVAL['rlhf_beta']['drift']:.3f} nats per response{NOTE}")
assert EVAL["rlhf_beta"]["drift"] > 0, "the policy did not move: is the optimizer stepping?"
if TRAINED_CHECKS:
    print(f"thresholds: {THRESHOLDS_SOURCE}")
    _ok = [check(gain >= THRESHOLDS["reward_gain_min"], f"reward gain {gain:.3f} < {THRESHOLDS['reward_gain_min']}"),
           check(EVAL["rlhf_beta"]["drift"] <= THRESHOLDS["drift_max"],
                 f"drift {EVAL['rlhf_beta']['drift']:.3f} > {THRESHOLDS['drift_max']}")]
    print("Checkpoint 3 passed" if all(_ok) else "Checkpoint 3: failures recorded for the seed protocol")
else:
    print("Checkpoint 3 (trained policy): SKIPPED on this runtime. Targets on a T4: reward gain >= "
          f"{THRESHOLDS['reward_gain_min']}, drift <= {THRESHOLDS['drift_max']} nats.")

**Explain.** If the reward-model score rose, the policy now samples more of the responses $r_\phi$ rates highly. The drift says what that cost: a drift of $k$ nats means that, averaged in log space, the policy makes its own responses $e^k$ times more likely than the reference does. The gold reward says whether the gain is real by the standard $r_\phi$ stands for: compare its change with the proxy's. In the training curves, every token now pays $\beta$ times its log-ratio, which slows the growth of the exact KL.

## Exercise 4 · Remove the Penalty: Reward Hacking (8 minutes)

Nothing new to write. The next cell reruns `rlhf_train` with `BETA_HACK = 0.0`: the same seed, steps, learning rate, prompts and evaluation generator. Only $\beta$ differs.

Remember what the gold rule contains that the reference's samples almost never show (Step 0): no credit beyond three distinct positive words, and a penalty when more than a quarter of the words come from its lists. The reward model was trained only on reference samples, so it never saw those parts of the rule. **The gold reward is a hidden rule we wrote**, known only because the data are synthetic; it is not human preference, and a real RLHF run has nothing like it.

**Predict.** Without the penalty, what happens to (a) the reward-model score, (b) the drift, (c) the gold reward, (d) the variety of the 8 samples per prompt? What do you expect the samples to look like?

In [ ]:
BETA_HACK = 0.0
started = time.time()
policy_hack, log_hack = rlhf_train(beta=BETA_HACK)
TIMES["RLHF, beta = 0"] = time.time() - started
EVAL["rlhf_beta0"] = evaluate(policy_hack)
print(f"\n{RUN['steps']} steps in {TIMES['RLHF, beta = 0']:.1f} s{NOTE}")
plot_training({f"beta = {BETA}": log_beta, "beta = 0": log_hack}, "With and without the KL penalty")

# Proxy and gold against drift, during training (the briefing's over-optimization figure, #fig-reward-drift).
fig, ax = plt.subplots(figsize=(6, 3.6))
for log, name, color in ((log_beta, f"beta = {BETA}", NAVY), (log_hack, "beta = 0", ACCENT)):
    ax.plot(log["kl_exact"], log["score"], "o-", ms=3, color=color, label=f"r_phi (proxy), {name}")
    ax.plot(log["kl_exact"], log["gold"], "s--", ms=3, color=color, alpha=0.6, label=f"gold, {name}")
ax.set_xlabel("batch mean exact KL (nats per response)")
ax.set_ylabel("batch mean reward")
ax.set_title("Proxy and gold reward against drift" + (" [TEST-ONLY stand-in]" if OFFLINE else ""))
ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
# Checkpoint 4: the reward-hacking signature, beta = 0 against beta = BETA (same seed)
workshop.checkpoint(label="4")
b, h = EVAL["rlhf_beta"], EVAL["rlhf_beta0"]
print(f"{'':<14}{'r_phi':>8}{'gold':>8}{'drift (nats)':>14}{'distinct-2':>12}")
for name in ("reference", "rlhf_beta", "rlhf_beta0"):
    e = EVAL[name]
    print(f"{name:<14}{e['score']:>+8.3f}{e['gold']:>+8.3f}{e['drift']:>14.3f}{e['distinct2']:>12.3f}")
_peak = int(np.argmax(log_hack["gold"]))
print(f"gold reward during the beta = 0 run: peak {log_hack['gold'][_peak]:+.3f} at step {log_hack['step'][_peak]}, "
      f"final {log_hack['gold'][-1]:+.3f} (batch means; printed, not asserted){NOTE}")

_i = 0  # the first evaluation prompt; its samples are rows 0 .. eval_samples-1
print(f"\nPROMPT: {PROMPT_TEXT[tuple(eval_prompt_ids[_i].tolist())]!r}")
for name in ("reference", "rlhf_beta", "rlhf_beta0"):
    print(f"{name}:")
    for text in decode(EVAL[name]["responses"][: min(4, RUN["eval_samples"])]):
        show(repr(text))

if TRAINED_CHECKS:
    T = THRESHOLDS
    print(f"\nthresholds: {THRESHOLDS_SOURCE}")
    _ok = [
        check(h["score"] >= b["score"] - T["hack_score_tol"],  # (a) the proxy held up
              f"(a) r_phi {h['score']:.3f} < {b['score']:.3f} - {T['hack_score_tol']}"),
        check(h["drift"] >= T["hack_drift_ratio"] * b["drift"] and h["drift"] >= T["hack_drift_floor"],  # (b)
              f"(b) drift {h['drift']:.2f} vs {b['drift']:.2f} (ratio {T['hack_drift_ratio']}, "
              f"floor {T['hack_drift_floor']})"),
        check(b["gold"] - h["gold"] >= T["hack_gold_margin"],  # (c) the gold reward fell
              f"(c) gold {h['gold']:.3f} not below {b['gold']:.3f} by {T['hack_gold_margin']}"),
        check(b["distinct2"] - h["distinct2"] >= T["hack_distinct_margin"],  # (d) the variety fell
              f"(d) distinct-2 {h['distinct2']:.3f} not below {b['distinct2']:.3f} by {T['hack_distinct_margin']}"),
    ]
    print("Checkpoint 4 passed: the reward-model score held up while the gold reward and the variety fell"
          if all(_ok) else "Checkpoint 4: failures recorded for the seed protocol")
else:
    print("\nCheckpoint 4 (trained policies): SKIPPED on this runtime. On a T4 it asserts, against the beta = BETA run: "
          "(a) r_phi at least as high, (b) drift at least 3 times larger and above a floor, (c) gold lower, (d) distinct-2 lower.")

**Explain.** Without the penalty nothing stops the policy from moving wherever $r_\phi$ is highest, including text unlike anything $r_\phi$ was trained on. Read the samples, and the gold reward's peak and final values printed above. The gold rule lets the gold reward part from $r_\phi$ in three ways:

- **No credit for more.** Only distinct positive words count, and at most three: repeats and a fourth word earn nothing, however highly $r_\phi$ scores them.
- **Crowding.** When more than a quarter of the words come from the lists, the gold reward falls.
- **Fewer positive words, or more negative ones.** The gold reward also falls, with neither of those parts firing, when the samples hold fewer distinct positive words than before, or more negative ones.

The reference's samples almost never trigger the first two, so $r_\phi$ never learned them (Step 0). The third needs no such blind spot: on text unlike its training data, $r_\phi$'s score is extrapolation and need not fall with the gold reward.

One exploratory run at full settings (seed 0, on the GPU of an Apple M1 Pro laptop; not the T4 seed protocol, and not Colab) went the third way. The `beta = 0` policy collapsed onto one positive word followed by filler, `good as as as …`, which the gold rule scores +1 with neither the cap nor the crowding penalty firing. During training its gold reward (batch mean) peaked at +1.92 at step 40 and fell to +1.00 while $r_\phi$ crept higher. On the evaluation prompts it scored +1.00 against +1.15 for the `beta = BETA` policy, a narrow gap for part (c). Your run may go another way.

Whichever way it went, when the proxy holds up while the gold reward falls, you are looking at **reward hacking**: Goodhart's law in a training loop (briefing, section 7). If the samples also become alike, distinct-2 falls: **mode collapse**. In the plot of reward against drift, watch where the two curves separate. Notice, too, what the gold rule misses: it scores `good as as as …` the same as a fluent sentence with one positive word. A rule we wrote is a better judge than $r_\phi$ only on what it was written to check. In a real system nobody wrote a rule down at all, which is why drift has to be watched and samples read.

# Part B · DPO

## Exercise 5 · The DPO Loss (12 minutes)

DPO fits the policy to the preference pairs directly (briefing, eqs. `dpo` and `implicit-reward`):

$$
\mathcal{L}_{\text{DPO}}(\theta) = -\,\mathbb{E}\Big[\log \sigma\big(\hat{r}_\theta(x, y_w) - \hat{r}_\theta(x, y_l)\big)\Big], \qquad \hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} .
$$

It is Lab 9's Bradley–Terry loss with the reward model replaced by the **implicit reward** $\hat{r}_\theta$. There is no reward model and no sampling: two networks (the policy and the frozen reference) and a fixed set of pairs.

**Predict.** Before any training $\pi_\theta = \pi_{\text{ref}}$. What is the loss? After a gradient step on one pair, which way do $\log \pi_\theta(y_w \mid x)$, $\log \pi_\theta(y_l \mid x)$ and their difference move?

**Task.** Complete `dpo_loss(logp_w, logp_l, logp_ref_w, logp_ref_l, beta)`. All inputs are `(N,)` sequence log-probabilities (sums over response tokens of `response_logprobs`). Return `(loss, reward_w, reward_l)`, with the two implicit rewards detached. Use `bt_loss` from Lab 9.

<details>
<summary>Hint for TODO 5</summary>

Each implicit reward is $\beta$ times that response's log-ratio of policy to reference, $\beta\,(\log \pi_\theta(y \mid x) - \log \pi_{\text{ref}}(y \mid x))$ (briefing eq. `implicit-reward`), and the loss is Lab 9's `bt_loss` of the preferred response's reward against the rejected one's. A common slip is $\beta\,(\log \pi_\theta(y_w \mid x) - \log \pi_\theta(y_l \mid x))$ with no reference terms: at $\pi_\theta = \pi_{\text{ref}}$ the loss must be exactly $\log 2$ for every pair.

</details>

In [ ]:
# TODO 5: complete this function.
def dpo_loss(logp_w, logp_l, logp_ref_w, logp_ref_l, beta):
    """All inputs (N,): sequence log-probs, i.e. sums over response tokens of response_logprobs."""
    raise NotImplementedError("TODO 5")

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def dpo_loss(logp_w, logp_l, logp_ref_w, logp_ref_l, beta):
    """All inputs (N,): sequence log-probs, i.e. sums over response tokens of response_logprobs."""
    reward_w = beta * (logp_w - logp_ref_w)          # implicit reward of the preferred response
    reward_l = beta * (logp_l - logp_ref_l)
    loss = bt_loss(reward_w, reward_l)               # = -F.logsigmoid(reward_w - reward_l).mean()
    return loss, reward_w.detach(), reward_l.detach()

<details>
<summary>Why this solution works</summary>

With the implicit rewards in place of $r_\phi$, `bt_loss` is exactly eq. `dpo`: `F.logsigmoid` computes $\log \sigma$ without overflow. At $\pi_\theta = \pi_{\text{ref}}$ both implicit rewards are 0, so the loss is $-\log \sigma(0) = \log 2$. The gradient is $-\beta\,\sigma(-\Delta)\,(\nabla \log \pi_\theta(y_w \mid x) - \nabla \log \pi_\theta(y_l \mid x))$ with $\Delta$ the reward difference: a step widens the gap between the two log-probabilities, weighted by how wrongly the pair is ranked now. The rewards are returned detached because they are for logging and accuracy, not for the gradient.

The checkpoint takes one small SGD step on a real pair. To first order the step always widens the margin $\log \pi_\theta(y_w \mid x) - \log \pi_\theta(y_l \mid x)$, so that is asserted. How each log-probability moves on its own depends on how similar the two gradients are, so those two numbers are only printed: both can move the same way (briefing, section 6, caution 3).

</details>

In [ ]:
# Checkpoint 5 (unit)
workshop.checkpoint(5)
_lw, _ll = torch.tensor([-10.0, -12.0]), torch.tensor([-11.0, -11.0])
_loss, _rw, _rl = dpo_loss(_lw, _ll, _lw.clone(), _ll.clone(), beta=0.1)
assert abs(_loss.item() - math.log(2)) < 1e-6, (
    f"identical policy and reference: loss {_loss.item()}, expected log 2 = {math.log(2):.6f}: "
    "each implicit reward is beta * (logp - logp_ref) of its own response")
assert torch.equal(_rw, torch.zeros(2)) and torch.equal(_rl, torch.zeros(2)), (
    f"at pi = pi_ref both implicit rewards are 0; got reward_w {_rw.tolist()} and reward_l {_rl.tolist()}")
_ref_w, _ref_l = torch.tensor([-10.5, -12.0]), torch.tensor([-10.0, -11.5])
_loss, _rw, _rl = dpo_loss(_lw, _ll, _ref_w, _ref_l, beta=0.5)
# Hand computation: reward_w = [0.25, 0.0], reward_l = [-0.5, 0.25], differences [0.75, -0.25].
_hand = (math.log(1 + math.exp(-0.75)) + math.log(1 + math.exp(0.25))) / 2
assert torch.allclose(_rw, torch.tensor([0.25, 0.0])) and torch.allclose(_rl, torch.tensor([-0.5, 0.25])), (
    f"expected reward_w [0.25, 0.0] and reward_l [-0.5, 0.25], got {_rw.tolist()} and {_rl.tolist()}: "
    "each is beta * (logp - logp_ref)")
assert abs(_loss.item() - _hand) < 1e-6, f"loss {_loss.item():.6f}, hand computation {_hand:.6f}"
assert not _rw.requires_grad and not _rl.requires_grad
_lw_g, _ll_g = _lw.clone().requires_grad_(True), _ll.clone().requires_grad_(True)
dpo_loss(_lw_g, _ll_g, _ref_w, _ref_l, beta=0.5)[0].backward()
assert (_lw_g.grad < 0).all() and (_ll_g.grad > 0).all(), "descending the loss must raise logp_w and lower logp_l"

# One SGD step on a fixed training pair, on a copy of the reference.
_x, _yw, _yl = (t[:1] for t in train_pairs[:3])
_model = copy.deepcopy(pi_ref).requires_grad_(True)
with torch.no_grad():
    _ref_w1, _ref_l1 = (response_logprobs(pi_ref, _x, y).sum(1) for y in (_yw, _yl))
_logp_w, _logp_l = response_logprobs(_model, _x, _yw).sum(1), response_logprobs(_model, _x, _yl).sum(1)
dpo_loss(_logp_w, _logp_l, _ref_w1, _ref_l1, beta=BETA)[0].backward()
with torch.no_grad():
    for _param in _model.parameters():
        _param -= 1e-4 * _param.grad  # plain SGD, small step
    _after_w, _after_l = (response_logprobs(_model, _x, y).sum(1).item() for y in (_yw, _yl))
_dw, _dl = _after_w - _logp_w.item(), _after_l - _logp_l.item()
print(f"after one step: logp_w {_dw:+.4f}, logp_l {_dl:+.4f}, margin logp_w - logp_l {_dw - _dl:+.4f}{NOTE}")
assert _dw - _dl > 0, "one step on a pair must widen the margin logp_w - logp_l"
del _model
print("Checkpoint 5 (unit) passed")

**Run: DPO training.** The provided `dpo_train` starts from a fresh copy of $\pi_{\text{ref}}$ and makes one pass over Lab 9's 8,000 training pairs, the same pairs the reward model was trained on, in batches of 32, with the same `BETA` as Exercise 3. The reference log-probabilities are computed once, before training: the reference is frozen, so they never change. Every few steps it logs the batch means of $\log \pi_\theta(y_w \mid x) - \log \pi_{\text{ref}}(y_w \mid x)$ and the same for $y_l$.

Afterwards we compute the **held-out preference accuracy** of the implicit reward on Lab 9's 1,000 held-out pairs: the share of pairs where $\hat{r}_\theta(x, y_w) > \hat{r}_\theta(x, y_l)$. Beside it: the reward model's accuracy on the same pairs, and the accuracy of the gold rule itself, the ceiling set by the noise in the Bradley–Terry labels.

**Predict.** Will both log-probabilities rise, both fall, or move apart?

In [ ]:
@torch.no_grad()
def sequence_logprobs(model, prompt_ids, response_ids, chunk=128):
    """log pi(y | x) for every row, in chunks, without gradient: (N,)."""
    return torch.cat([response_logprobs(model, prompt_ids[i : i + chunk], response_ids[i : i + chunk]).sum(1)
                      for i in range(0, len(prompt_ids), chunk)])


def preference_accuracy(reward_w, reward_l):
    """Share of pairs ranked the right way; a tie counts one half."""
    return ((reward_w > reward_l).float() + 0.5 * (reward_w == reward_l).float()).mean().item()


def dpo_train(beta, seed=SEED, lr=None, batch=None, epochs=1, log_every=None):
    """One pass of DPO over train_pairs from a fresh copy of pi_ref. Returns (policy, log)."""
    lr, batch = lr or DPO_LR, batch or RUN["dpo_batch"]
    x, y_w, y_l = train_pairs[:3]
    ref_w, ref_l = sequence_logprobs(pi_ref, x, y_w), sequence_logprobs(pi_ref, x, y_l)  # once: pi_ref is frozen
    torch.manual_seed(seed)
    policy = copy.deepcopy(pi_ref).requires_grad_(True)
    opt = torch.optim.AdamW(policy.parameters(), lr=lr, weight_decay=0.0)
    order = torch.cat([torch.randperm(len(x), generator=torch.Generator().manual_seed(seed + e)) for e in range(epochs)])
    n_steps = len(order) // batch
    log_every = log_every or max(1, n_steps // 25)
    log = {"step": [], "loss": [], "dlogp_w": [], "dlogp_l": [], "logp_w": [], "logp_l": []}
    started = time.time()
    for step in range(n_steps):
        idx = order[step * batch : (step + 1) * batch].to(DEVICE)
        # One forward pass for both responses of every pair: (2 * batch, L).
        logp = response_logprobs(policy, torch.cat([x[idx], x[idx]]), torch.cat([y_w[idx], y_l[idx]])).sum(1)
        logp_w, logp_l = logp[: len(idx)], logp[len(idx) :]
        loss, reward_w, reward_l = dpo_loss(logp_w, logp_l, ref_w[idx], ref_l[idx], beta)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), CLIP)
        opt.step()
        if step % log_every == 0 or step == n_steps - 1:
            log["step"].append(step)
            log["loss"].append(loss.item())
            log["dlogp_w"].append((logp_w - ref_w[idx]).mean().item())
            log["dlogp_l"].append((logp_l - ref_l[idx]).mean().item())
            log["logp_w"].append(logp_w.mean().item())
            log["logp_l"].append(logp_l.mean().item())
            print(f"step {step:>4}  loss {loss.item():.4f}  logp_w - ref {log['dlogp_w'][-1]:+7.3f}  "
                  f"logp_l - ref {log['dlogp_l'][-1]:+7.3f}  ({time.time() - started:.0f} s)")
    policy.eval().requires_grad_(False)
    return policy, log


started = time.time()
policy_dpo, log_dpo = dpo_train(beta=BETA)
TIMES["DPO"] = time.time() - started
EVAL["dpo"] = evaluate(policy_dpo)
print(f"\n{len(log_dpo['step']) and log_dpo['step'][-1] + 1} steps of batch {RUN['dpo_batch']} in {TIMES['DPO']:.1f} s{NOTE}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(log_dpo["step"], log_dpo["dlogp_w"], "o-", ms=3, color=GREEN, label="log pi(y_w|x) - log pi_ref(y_w|x)")
ax.plot(log_dpo["step"], log_dpo["dlogp_l"], "s-", ms=3, color=ACCENT, label="log pi(y_l|x) - log pi_ref(y_l|x)")
ax.axhline(0, color=GREY, linewidth=0.8)
ax.set_xlabel("step")
ax.set_ylabel("batch mean change (nats)")
ax.set_title("DPO: where the log-probabilities go" + (" [TEST-ONLY stand-in]" if OFFLINE else ""))
ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
# Checkpoint 5 (after training): held-out preference accuracy, and the final table
workshop.checkpoint(5, label="5 (after training)")
started = time.time()
hx, hw, hl, hg_w, hg_l = heldout_pairs
_ref_hw, _ref_hl = sequence_logprobs(pi_ref, hx, hw), sequence_logprobs(pi_ref, hx, hl)
ACC = {
    "reference": preference_accuracy(score(hx, hw), score(hx, hl)),  # r_phi on the same pairs
    "dpo": preference_accuracy(BETA * (sequence_logprobs(policy_dpo, hx, hw) - _ref_hw),
                               BETA * (sequence_logprobs(policy_dpo, hx, hl) - _ref_hl)),
    "gold rule": preference_accuracy(hg_w, hg_l),  # the ceiling: the labels are noisy draws from it
}
TIMES["held-out accuracy"] = time.time() - started
print(f"held-out preference accuracy on {len(hx):,} pairs: DPO implicit reward {ACC['dpo']:.3f} · "
      f"reward model {ACC['reference']:.3f} · gold rule {ACC['gold rule']:.3f}{NOTE}")

TIMES["training, total"] = TIMES["RLHF, beta = BETA"] + TIMES["RLHF, beta = 0"] + TIMES["DPO"]
ROWS = [("pi_ref", "reference", None), (f"RLHF, beta = {BETA}", "rlhf_beta", "RLHF, beta = BETA"),
        ("RLHF, beta = 0", "rlhf_beta0", "RLHF, beta = 0"), (f"DPO, beta = {BETA}", "dpo", "DPO")]
print(f"\n{'policy':<20}{'r_phi':>8}{'gold':>8}{'drift (nats)':>14}{'distinct-2':>12}{'held-out acc':>14}{'train (s)':>11}")
for label, key, timer in ROWS:
    e = EVAL[key]
    acc = f"{ACC[key]:.3f}" if key in ACC else "–"
    secs = f"{TIMES[timer]:.0f}" if timer else "–"
    print(f"{label:<20}{e['score']:>+8.3f}{e['gold']:>+8.3f}{e['drift']:>14.3f}{e['distinct2']:>12.3f}{acc:>14}{secs:>11}")
print(f"(evaluation: {RUN['eval_prompts']} prompts x {RUN['eval_samples']} samples; held-out accuracy: implicit "
      f"reward for DPO, r_phi for pi_ref){NOTE}")

if TRAINED_CHECKS:
    _ok = check(ACC["dpo"] >= THRESHOLDS["dpo_acc_min"], f"DPO held-out accuracy {ACC['dpo']:.3f} < {THRESHOLDS['dpo_acc_min']}")
    print("Checkpoint 5 passed" if _ok else "Checkpoint 5: failure recorded for the seed protocol")
else:
    print(f"Checkpoint 5 (trained policy): SKIPPED on this runtime. Target on a T4: accuracy >= {THRESHOLDS['dpo_acc_min']}.")

**Explain.** Read the table row by row.

- **DPO** reached its held-out accuracy without a reward model or a sampling loop: its implicit reward is a reward model you can evaluate like Lab 9's (the paper's subtitle). Compare it with $r_\phi$'s accuracy and with the gold rule's ceiling.
- **Drift.** DPO never sees its own samples, and nothing in its loss measures their KL from the reference. Its drift is whatever the pass over the data produced (briefing, section 6, caution 2).
- **The log-probability plot.** The DPO loss sees only the difference of the two log-ratios, so both can fall while the rejected one falls faster (caution 3). The probability that leaves them goes to other responses.
- **No winner is asserted.** Whether DPO or RLHF does better depends on the budget, the data and $\beta$; published comparisons disagree.

## Results Card

Every number below was measured in this run. The cell also writes them to `lab10_results.json`.

In [ ]:
TIMES["total so far"] = time.time() - LAB_START
_prompt0 = PROMPT_TEXT[tuple(eval_prompt_ids[0].tolist())]
results = {
    "offline_test_stand_in": OFFLINE or None,  # if set, nothing here is a result
    "seed": SEED, "protocol": PROTOCOL, "device": DEVICE, "fast": FAST, "run": RUN, "hp": HP,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "gpu_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE == "cuda" else None,
    "model": CAUSAL_LM if not OFFLINE else MODEL_SOURCE, "revision": CAUSAL_LM_REVISION,
    "reward_model_file": os.environ.get("NLP_LLMS_LAB10_REWARD_MODEL") or LAB09_FILES["reward_model"][0],
    "thresholds": THRESHOLDS, "thresholds_source": THRESHOLDS_SOURCE, "check_failures": CHECK_FAILURES,
    "policies": {k: {m: round(v[m], 4) for m in ("score", "gold", "drift", "distinct2")} for k, v in EVAL.items()},
    "heldout_accuracy": {k: round(v, 4) for k, v in ACC.items()},
    "heldout_pairs": len(hx),
    "beta0_gold": {"peak": log_hack["gold"][_peak], "peak_step": log_hack["step"][_peak], "final": log_hack["gold"][-1]},
    "logs": {"rlhf_beta": log_beta, "rlhf_beta0": log_hack, "dpo": log_dpo},
    "samples": {"prompt": _prompt0, **{k: decode(v["responses"][: min(4, RUN["eval_samples"])]) for k, v in EVAL.items()}},
    "seconds": {k: round(v, 1) for k, v in TIMES.items()},
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "cuda": torch.version.cuda},
    "date": time.strftime("%Y-%m-%d"),
}
RESULTS_PATH = Path(os.environ.get("NLP_LLMS_LAB10_RESULTS", "lab10_results.json"))
RESULTS_PATH.write_text(json.dumps(results, indent=2))
print(json.dumps({k: v for k, v in results.items() if k not in ("logs", "samples")}, indent=2))
print("\nRun time by stage:")
for stage, seconds in TIMES.items():
    print(f"  {stage:<22} {seconds:>7.1f} s")
print("\nThis is the end of the core path." + NOTE)

## Stretch (optional) · The Reward–Drift Trade-Off

For those who finish early. Nothing later in the workshop depends on it.

The objective's optimum is the reference reweighted by $e^{r/\beta}$ (briefing, eq. `optimal-policy`): small $\beta$ lets the policy move far, large $\beta$ keeps it close. Sweep $\beta$ over five values, $0$, $\beta/4$, $\beta$, $4\beta$ and $16\beta$, with the same seed, each starting from $\pi_{\text{ref}}$. The runs of Exercises 3 and 4 are reused for $\beta$ and $0$; the three new runs are shorter (`RUN["sweep_steps"]` steps) to fit the time budget, and the plot marks them. Then plot mean $r_\phi$ and mean gold against drift, one point per $\beta$, with DPO's point added.

**Predict.** As $\beta$ falls from $16\beta$ to 0, how do the proxy and the gold reward move against drift? Where on the drift axis do they separate?

In [ ]:
SWEEP = {0.0: (EVAL["rlhf_beta0"], RUN["steps"]), BETA: (EVAL["rlhf_beta"], RUN["steps"])}
if RUN["sweep_steps"] and not SKIP_STRETCH:
    started = time.time()
    for beta in (BETA / 4, 4 * BETA, 16 * BETA):
        _policy, _ = rlhf_train(beta=beta, steps=RUN["sweep_steps"], verbose=False)
        SWEEP[beta] = (evaluate(_policy), RUN["sweep_steps"])
        print(f"beta = {beta:<7g} r_phi {SWEEP[beta][0]['score']:+.3f}  gold {SWEEP[beta][0]['gold']:+.3f}  "
              f"drift {SWEEP[beta][0]['drift']:.3f}{NOTE}")
        del _policy
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    TIMES["stretch"] = time.time() - started
    print(f"({TIMES['stretch']:.1f} s for 3 runs of {RUN['sweep_steps']} steps)")
    betas = sorted(SWEEP)
    points = [{"beta": beta, "steps": SWEEP[beta][1],
               **{m: round(SWEEP[beta][0][m], 4) for m in ("score", "gold", "drift", "distinct2")}} for beta in betas]
    points.append({"beta": BETA, "steps": None, "method": "DPO",
                   **{m: round(EVAL["dpo"][m], 4) for m in ("score", "gold", "drift", "distinct2")}})
    POINTS_PATH = Path("lab10_reward_drift_points.json")  # for the briefing's over-optimization figure (also in the results file)
    POINTS_PATH.write_text(json.dumps({"seed": SEED, "offline_test_stand_in": OFFLINE or None, "points": points}, indent=2))
    results["stretch"] = points
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    print(f"wrote {POINTS_PATH}")

    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    drifts = [SWEEP[b_][0]["drift"] for b_ in betas]
    ax.plot(drifts, [SWEEP[b_][0]["score"] for b_ in betas], "o-", color=NAVY, label="r_phi (proxy)")
    ax.plot(drifts, [SWEEP[b_][0]["gold"] for b_ in betas], "s--", color=ACCENT, label="gold reward")
    for b_ in betas:
        e, steps = SWEEP[b_]
        ax.annotate(f"beta={b_:g}" + ("" if steps == RUN["steps"] else f" ({steps} steps)"), (e["drift"], e["score"]),
                    textcoords="offset points", xytext=(4, 6), fontsize=7)
    ax.plot([EVAL["dpo"]["drift"]] * 2, [EVAL["dpo"]["score"], EVAL["dpo"]["gold"]], "D", color=GREEN, label="DPO (proxy, gold)")
    ax.set_xlabel("drift: mean exact KL (nats per response)")
    ax.set_ylabel("mean reward, evaluation prompts")
    ax.set_title(f"The reward-drift trade-off ({RUN['steps']} steps; new runs {RUN['sweep_steps']})"
                 + (" [TEST-ONLY stand-in]" if OFFLINE else ""), fontsize=9)
    ax.legend(frameon=False, fontsize=8)
    plt.show()

    monotone = all(SWEEP[lo][0]["drift"] >= SWEEP[hi][0]["drift"] for lo, hi in zip(betas, betas[1:]))
    print(f"drift falls as beta rises: {monotone}")
    if TRAINED_CHECKS and THRESHOLDS["sweep_monotone"]:
        check(monotone, "drift does not fall monotonically as beta rises")
else:
    print("Stretch SKIPPED: " + ("--skip-stretch was set." if SKIP_STRETCH else
                                 "on a CPU runtime, three more trainings would take far longer than 10 minutes."))

**Explain.** Each point is one value of $\beta$. Large $\beta$ keeps the policy near the reference: little drift, a small gain on both rewards. As $\beta$ falls, drift grows. Gao et al. found, with a large gold reward model, that the proxy keeps rising with drift while the gold reward rises, peaks and falls once the policy reaches text where the proxy is wrong. Compare your points with that shape, and note the three shorter runs. DPO's point shows where an offline method with the same $\beta$ ended up after one pass over the data.

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [11 · Calibration](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/11-calibration.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT